# ENMV Model comparison (WAIC, PSIS-LOO, and BIC)

This notebook compares fitted host-adaptation models with phenotype dimension \(n=1,2,3\) using posterior samples from MCMC inference. Model predictive performance is evaluated using **WAIC**, **PSIS-LOO cross-validation**, and an approximate **BIC**.

## Required files

The following files must be located in the same directory as this notebook:

- `Data_RESISTE_ENMV.xlsx` – experimental ENMV cross-inoculation data  
- `posterior_n1.pkl` – posterior samples for \(n=1\)  
- `posterior_n2.pkl` – posterior samples for \(n=2\)  
- `posterior_n3.pkl` – posterior samples for \(n=3\)


In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy import stats
import math

import warnings
warnings.filterwarnings("ignore", message="Workbook contains no default style")

# ============================================================
# 0) DATA LOADING (standalone): rebuild HOSTS, H, Y, Y_CL, N_CL
# ============================================================
DATA_RESISTE = "Data_RESISTE_ENMV.xlsx"
assert os.path.exists(DATA_RESISTE), f"Missing {DATA_RESISTE}. Put it next to this notebook."

# Hosts (must match the order used in inference)
HOSTS = ["CH", "LA", "SA", "SO", "ZI"]
H = len(HOSTS)
host_to_idx = {h: i for i, h in enumerate(HOSTS)}

# Excel columns -> host codes
host_map = {
    "chicorée": "CH",
    "laitue": "LA",
    "salsifis": "SA",
    "souci": "SO",
    "zinnia": "ZI",
}

raw = pd.read_excel(DATA_RESISTE, sheet_name="ENMV")

needed = ["sp_inoc", "soucheV", "plante_source", "infected"]
missing = [c for c in needed if c not in raw.columns]
assert not missing, f"Missing columns: {missing}. Columns found: {list(raw.columns)}"

raw["target"]  = raw["sp_inoc"].map(host_map)
raw["source"]  = raw["soucheV"].astype(str)
raw["lineage"] = raw["plante_source"].astype(str)
raw["y"]       = raw["infected"].astype(int)

valid_sources = ["7098MP1"] + HOSTS
raw = raw[raw["source"].isin(valid_sources)].copy()
raw = raw[raw["target"].isin(HOSTS)].copy()

# Split clonal vs evolved
df_clonal = raw[raw["source"] == "7098MP1"].copy()
df_evol   = raw[raw["source"] != "7098MP1"].copy()

# Clonal aggregated counts per target
clonal_counts = df_clonal.groupby("target")["y"].agg(["sum", "count"]).reindex(HOSTS)
assert clonal_counts["count"].min() > 0, "Some clonal targets have zero observations (check Excel filtering)."

Y_CL = clonal_counts["sum"].astype(int).values
N_CL = clonal_counts["count"].astype(int).values

# Evolved lineage-level: each (source,lineage,target) has n=12
lin = df_evol.groupby(["source", "lineage", "target"])["y"].agg(["sum", "count"]).reset_index()
lin.rename(columns={"sum": "succ", "count": "n"}, inplace=True)
assert lin["n"].min() == 12 and lin["n"].max() == 12, "Expected 12 plants per (source,lineage,target)."

# Enforce 8 lineages per source (consistent order)
lineages_by_source = {s: sorted(lin[lin["source"] == s]["lineage"].unique()) for s in HOSTS}
for s in HOSTS:
    assert len(lineages_by_source[s]) == 8, f"Expected 8 lineages for source {s}, got {len(lineages_by_source[s])}"

# Build array Y[src, tgt, lineage] = successes (0..12)
Y = np.zeros((H, H, 8), dtype=int)
for src in HOSTS:
    for si, lin_id in enumerate(lineages_by_source[src]):
        sub = lin[(lin["source"] == src) & (lin["lineage"] == lin_id)]
        for tgt in HOSTS:
            y = int(sub[sub["target"] == tgt]["succ"].iloc[0])
            Y[host_to_idx[src], host_to_idx[tgt], si] = y

print("Data loaded:")
print("  HOSTS:", HOSTS)
print("  Y shape:", Y.shape, "(H,H,8)")
print("  Y_CL:", Y_CL, "N_CL:", N_CL)

ER_jk = Y.sum(axis=2)                       # shape (H,H), aggregated successes over 8 lineages
denom = Y.shape[2] * 12.0                   # 8 * 12 = 96
p_hat = ER_jk / denom
QK = p_hat.max(axis=0)                      # shape (H,), max over sources for each target

print("q_k (max per target over evolved sources):")
for k, name in enumerate(HOSTS):
    print(f"  q_{name} = {QK[k]:.6f}")

# ============================================================
# 1) USER SETTINGS: filenames + options
# ============================================================
PKL_FILES = {
    1: "posterior_n1.pkl",
    2: "posterior_n2.pkl",
    3: "posterior_n3.pkl",
}

KAPPA_DEFAULT = 10.0
MAX_DRAWS = 4000
RNG_SEED = 123

# ============================================================
# 2) Load posterior pickles
# ============================================================
def load_posterior(path):
    assert os.path.exists(path), f"Missing posterior file: {path}"
    with open(path, "rb") as f:
        post = pickle.load(f)

    chains = post.get("chains", None)
    if chains is None:
        chains = post.get("results", None)
    assert chains is not None, f"Couldn't find 'chains' or 'results' in {path}"

    kappa = float(post.get("kappa", KAPPA_DEFAULT))

    n_dim = post.get("n_dim", None)
    assert n_dim is not None, f"Missing 'n_dim' in {path}"
    n_dim = int(n_dim)

    return post, chains, kappa, n_dim

def stack_samples(chains):
    return np.vstack([c["samples"] for c in chains])

rng = np.random.default_rng(RNG_SEED)
def maybe_subsample(S):
    if MAX_DRAWS is None or S.shape[0] <= MAX_DRAWS:
        return S
    idx = rng.choice(S.shape[0], size=MAX_DRAWS, replace=False)
    return S[idx]

# Load all models
models = {}
print("\nLoaded:")
for n_dim_expected, path in PKL_FILES.items():
    post, chains, kappa, n_dim = load_posterior(path)
    print(f"  n={n_dim_expected}: {path}  n_dim={n_dim}  kappa={kappa}  chains={len(chains)}")
    assert n_dim == n_dim_expected, f"{path}: expected n_dim={n_dim_expected}, got {n_dim}"
    S = maybe_subsample(stack_samples(chains))
    models[n_dim] = {"post": post, "chains": chains, "kappa": kappa, "S": S}

# ============================================================
# 3) Mechanistic mapping (must match inference / notebook)
# ============================================================
EPS_P = 1e-6
EPS_DEN = 1e-8

def p_from_d2(d2, log10_N0U, R, n_dim, eps_p=EPS_P, eps_den=EPS_DEN):
    R2 = R * R
    if d2 < R2:
        return 1.0 - eps_p

    denom = d2 / R2 - 1.0
    if denom < eps_den:
        denom = eps_den

    mu = 1.0 - (d2 + n_dim) / R2
    sig2 = (4.0 * d2 + 2.0 * n_dim) / (R2 * R2)
    sig = math.sqrt(max(sig2, 1e-12))
    z = mu / sig

    phi = stats.norm.pdf(z)
    Phi = stats.norm.cdf(z)
    trunc_mean = sig * phi + mu * Phi
    trunc_mean = max(trunc_mean, 0.0)

    rate = (10.0 ** log10_N0U) * (2.0 / denom) * trunc_mean
    p = 1.0 - math.exp(-rate)
    p = min(max(p, eps_p), 1.0 - eps_p)
    return p

def theta_to_params(theta, n_dim):
    # theta = (log10_N0U, logR_k (k=1..H), O_flat)
    log10_N0U = float(theta[0])
    logRk = np.asarray(theta[1:1 + H], float)
    Rk = np.exp(logRk)
    O = np.asarray(theta[1 + H:], float).reshape(H, n_dim)
    return log10_N0U, Rk, O

def build_p_matrix(O, log10_N0U, Rk, n_dim, qk=None):
    # O: (H, n_dim) optima for CH,LA,SA,SO,ZI (HOSTS order)
    # Rk: one geometric scale R_k per *target* host (HOSTS order)
    # qk: fixed target-specific success ceilings (HOSTS order), NOT sampled
    if np.isscalar(Rk):
        Rk = np.full(H, float(Rk))
    else:
        Rk = np.asarray(Rk, float).reshape(-1)
        assert Rk.size == H, f"Expected Rk of length {H}, got {Rk.size}"

    if qk is None:
        qk = QK
    qk = np.asarray(qk, float).reshape(-1)
    assert qk.size == H, f"Expected qk of length {H}, got {qk.size}"

    p_cl = np.zeros(H, float)
    p_evol = np.zeros((H, H), float)

    # CL is at origin, so d2 = ||O_k||^2
    for k in range(H):
        d2 = float(np.dot(O[k], O[k]))
        p = p_from_d2(d2, log10_N0U, float(Rk[k]), n_dim)
        p = float(qk[k]) * p
        p_cl[k] = min(max(p, EPS_P), 1.0 - EPS_P)

    # evolved: distances between optima
    for j in range(H):
        for k in range(H):
            d = O[j] - O[k]
            d2 = float(np.dot(d, d))
            p = p_from_d2(d2, log10_N0U, float(Rk[k]), n_dim)
            p = float(qk[k]) * p
            p_evol[j, k] = min(max(p, EPS_P), 1.0 - EPS_P)

    return p_cl, p_evol

# ============================================================
# 4) Pointwise log-likelihood + matrices
# ============================================================
def loglik_pointwise(theta, n_dim, kappa):
    log10_N0U, Rk, O = theta_to_params(theta, n_dim)

    # IMPORTANT: include fixed q_k in the generative probabilities (matches your notebook)
    p_cl, p_evol = build_p_matrix(O, log10_N0U, Rk, n_dim, qk=QK)

    ll_parts = []

    # Clonal block: 5 binom observations (one per target host)
    ll_cl = stats.binom.logpmf(Y_CL, N_CL, np.clip(p_cl, EPS_P, 1.0 - EPS_P))
    ll_parts.append(ll_cl.astype(float))

    # Evolved block: H*H*8 beta-binomial observations (one per lineage)
    ll_lin = []
    for j in range(H):
        for k in range(H):
            p = float(np.clip(p_evol[j, k], EPS_P, 1.0 - EPS_P))
            a = kappa * p
            b = kappa * (1.0 - p)
            ll_lin.append(stats.betabinom.logpmf(Y[j, k, :], 12, a, b))
    ll_parts.append(np.concatenate(ll_lin).astype(float))

    return np.concatenate(ll_parts)

def compute_loglik_matrix(S, n_dim, kappa):
    return np.vstack([loglik_pointwise(S[i], n_dim, kappa) for i in range(S.shape[0])])

for n_dim, m in models.items():
    m["L"] = compute_loglik_matrix(m["S"], n_dim, m["kappa"])
    print(f"\nloglik matrix n={n_dim}:", m["L"].shape)

# ============================================================
# 5) WAIC
# ============================================================
def logmeanexp(a, axis=0):
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    return (m + np.log(np.mean(np.exp(a - m), axis=axis, keepdims=True))).squeeze(axis)

def waic_from_loglik(L):
    # L: (n_draws, n_obs) matrix of pointwise log-likelihoods
    lppd_i = logmeanexp(L, axis=0)
    p_waic_i = np.var(L, axis=0, ddof=1)
    elpd_i = lppd_i - p_waic_i
    elpd = float(np.sum(elpd_i))
    se = float(np.sqrt(L.shape[1] * np.var(elpd_i, ddof=1)))
    waic = float(-2.0 * elpd)
    return {"elpd": elpd, "se": se, "waic": waic, "elpd_i": elpd_i}

for n_dim, m in models.items():
    m["waic"] = waic_from_loglik(m["L"])

print("\nWAIC:")
for n_dim in sorted(models):
    w = models[n_dim]["waic"]
    print(f"  n={n_dim}: elpd={w['elpd']:.2f}  SE={w['se']:.2f}  WAIC={w['waic']:.2f}")

# Pairwise delta elpd_waic
def delta_summary(a, b, name="Δelpd"):
    d_i = b["elpd_i"] - a["elpd_i"]
    d = float(np.sum(d_i))
    se = float(np.sqrt(d_i.size * np.var(d_i, ddof=1)))
    return d, se

print("\nPairwise Δelpd_waic (B - A):")
pairs = [(1, 2), (1, 3), (2, 3)]
for a, b in pairs:
    d, se = delta_summary(models[a]["waic"], models[b]["waic"], "Δelpd_waic")
    print(f"  n={b} - n={a}: {d:.2f}  SE={se:.2f}")
# ============================================================
# 6) PSIS-LOO (standalone, no ArviZ dependency)
# ============================================================

def _logsumexp(x, axis=None):
    x = np.asarray(x, float)
    m = np.max(x, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return out.squeeze(axis)

def psis_smooth_weights(logw, tail_frac=0.2, min_tail=10):
    """
    Pareto-smoothed importance sampling (PSIS) for one set of log-weights.

    Parameters
    ----------
    logw : array, shape (S,)
        Raw log-weights (unnormalized), typically logw = -loglik_i
    tail_frac : float
        Fraction of largest weights used as tail (default 0.2).
    min_tail : int
        Minimum number of tail samples for the GPD fit.

    Returns
    -------
    logw_smooth : array, shape (S,)
        Smoothed log-weights (still unnormalized).
    k_hat : float
        Estimated Pareto shape parameter k (diagnostic).
    """
    lw = np.asarray(logw, float).copy()
    S = lw.size
    if S < 5:
        # too few draws: return as-is
        return lw, np.nan

    # Stabilize
    lw -= np.max(lw)
    w = np.exp(lw)

    # Tail size
    M = int(np.ceil(tail_frac * S))
    M = max(M, min_tail)
    M = min(M, S - 1)  # keep at least one non-tail point

    # Sort weights
    idx = np.argsort(w)
    w_sorted = w[idx]
    tail_idx = idx[-M:]
    w_tail = w[tail_idx]

    # Threshold u: smallest tail weight
    u = w_sorted[-M]
    excess = w_tail - u
    excess = np.maximum(excess, 0.0)

    # If tail has no spread, nothing to fit
    if np.all(excess <= 0) or np.max(excess) <= 0:
        return lw, 0.0

    # Fit Generalized Pareto to excesses with loc=0
    # SciPy parameterization: genpareto(c=k, loc=0, scale=σ)
    try:
        k_hat, loc_hat, sig_hat = stats.genpareto.fit(excess, floc=0.0)
        # Guard against weird fits
        if not np.isfinite(k_hat) or not np.isfinite(sig_hat) or sig_hat <= 0:
            return lw, np.nan
    except Exception:
        return lw, np.nan

    # Replace tail weights by expected order stats from fitted GPD
    # Use plotting positions (j - 0.5)/M for j=1..M
    j = np.arange(1, M + 1)
    p = (j - 0.5) / M
    # Quantiles of excesses
    q_excess = stats.genpareto.ppf(p, c=k_hat, loc=0.0, scale=sig_hat)
    q_excess = np.maximum(q_excess, 0.0)
    w_tail_smooth = u + q_excess

    # Truncate extreme weights (helps when k>1 etc.)
    w_max = np.max(w)
    w_tail_smooth = np.minimum(w_tail_smooth, w_max)

    # Put back into full weight vector
    w_smooth = w.copy()
    # assign smoothed tail in ascending order to match ranks
    tail_sorted_order = np.argsort(w_tail)  # ascending within tail
    w_smooth[tail_idx[tail_sorted_order]] = w_tail_smooth

    # Back to log space (still unnormalized)
    logw_smooth = np.log(np.maximum(w_smooth, 1e-300))
    return logw_smooth, float(k_hat)

def psis_loo_from_loglik(L, tail_frac=0.2):
    """
    Compute PSIS-LOO from a log-likelihood matrix L (draws x obs).

    Returns dict with:
      elpd_loo, se, elpd_loo_i, pareto_k
    """
    L = np.asarray(L, float)
    S, N = L.shape
    elpd_i = np.zeros(N, float)
    pareto_k = np.zeros(N, float)

    for i in range(N):
        ll_i = L[:, i]

        # Raw log-weights for LOO: w_s ∝ 1 / p(y_i | θ_s)
        logw_raw = -ll_i

        logw_smooth, k_hat = psis_smooth_weights(logw_raw, tail_frac=tail_frac)
        pareto_k[i] = k_hat

        # Normalize smoothed weights
        logw_norm = logw_smooth - _logsumexp(logw_smooth, axis=0)

        # LOO predictive density: sum_s w_s * p(y_i | θ_s)
        # in log space: logsumexp(logw_norm + ll_i)
        elpd_i[i] = _logsumexp(logw_norm + ll_i, axis=0)

    elpd = float(np.sum(elpd_i))
    se = float(np.sqrt(N * np.var(elpd_i, ddof=1)))
    looic = float(-2.0 * elpd)
    return {"elpd": elpd, "se": se, "looic": looic, "elpd_i": elpd_i, "pareto_k": pareto_k}

# Compute PSIS-LOO for each model
for n_dim, m in models.items():
    m["loo"] = psis_loo_from_loglik(m["L"], tail_frac=0.2)

print("\nPSIS-LOO (standalone):")
for n_dim in sorted(models):
    loo = models[n_dim]["loo"]
    print(f"  n={n_dim}: elpd={loo['elpd']:.2f}  SE={loo['se']:.2f}  LOOIC={loo['looic']:.2f}")

print("\nPairwise Δelpd_loo (B - A):")
pairs = [(1, 2), (1, 3), (2, 3)]
for a, b in pairs:
    d_i = models[b]["loo"]["elpd_i"] - models[a]["loo"]["elpd_i"]
    d = float(np.sum(d_i))
    se = float(np.sqrt(d_i.size * np.var(d_i, ddof=1)))
    print(f"  n={b} - n={a}: {d:.2f}  SE={se:.2f}")

print("\nPareto k summaries (diagnostics):")
for n_dim in sorted(models):
    k = models[n_dim]["loo"]["pareto_k"]
    print(f"  n={n_dim}: max={float(np.nanmax(k)):.3f}  >0.7={int(np.sum(k > 0.7))}/{k.size}  >1.0={int(np.sum(k > 1.0))}/{k.size}")

# ============================================================
# 7) OPTIONAL: BIC@best-draw (approx)
# ============================================================
def bic_at_best_draw(L, n_params):
    ll_tot = np.sum(L, axis=1)
    ll_max = float(np.max(ll_tot))
    N = int(L.shape[1])
    bic = float(-2.0 * ll_max + n_params * math.log(N))
    return ll_max, bic

print("\nBIC@best-draw (approx, data-only):")
for n_dim in sorted(models):
    # q_k is fixed (not estimated), so parameter count is unchanged
    k_params = 1 + H + H * n_dim
    ll_max, bic = bic_at_best_draw(models[n_dim]["L"], k_params)
    print(f"  n={n_dim}: ll_max={ll_max:.2f}  k={k_params}  BIC={bic:.2f}")


Data loaded:
  HOSTS: ['CH', 'LA', 'SA', 'SO', 'ZI']
  Y shape: (5, 5, 8) (H,H,8)
  Y_CL: [19 16 21  0  4] N_CL: [20 20 40 60 20]
q_k (max per target over evolved sources):
  q_CH = 1.000000
  q_LA = 0.885417
  q_SA = 0.614583
  q_SO = 1.000000
  q_ZI = 0.927083

Loaded:
  n=1: posterior_n1.pkl  n_dim=1  kappa=10.0  chains=10
  n=2: posterior_n2.pkl  n_dim=2  kappa=10.0  chains=10
  n=3: posterior_n3.pkl  n_dim=3  kappa=10.0  chains=10

loglik matrix n=1: (4000, 205)

loglik matrix n=2: (4000, 205)

loglik matrix n=3: (4000, 205)

WAIC:
  n=1: elpd=-982.41  SE=116.37  WAIC=1964.82
  n=2: elpd=-389.87  SE=32.16  WAIC=779.75
  n=3: elpd=-391.17  SE=33.14  WAIC=782.33

Pairwise Δelpd_waic (B - A):
  n=2 - n=1: 592.53  SE=110.99
  n=3 - n=1: 591.24  SE=110.92
  n=3 - n=2: -1.29  SE=15.62

PSIS-LOO (standalone):
  n=1: elpd=-788.98  SE=75.86  LOOIC=1577.97
  n=2: elpd=-376.34  SE=27.01  LOOIC=752.69
  n=3: elpd=-385.08  SE=28.30  LOOIC=770.16

Pairwise Δelpd_loo (B - A):
  n=2 - n=1: 412.64